<a href="https://colab.research.google.com/github/Aljwharah-h/AI-Agents-System/blob/main/Assignment2_Create_an_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies

In [4]:
%pip install -q langchain langchain_openai langchain_community

In [5]:
import os

Set API Keys

In [6]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY") # Ensure Tavily API key is loaded
except ImportError:
    pass

Create Tools

In [7]:
# This tool to fetch content from any URL
import requests
from langchain.tools import tool

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text

In [8]:
!pip install tavily-python

In [9]:
from google.colab import userdata
from tavily import TavilyClient

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY") # Use the correct environment variable
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [10]:
from typing import Literal

# This tool used for internet search

@tool
def internet_search(
    query: str,
    max_results: int = 3,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [11]:
result = internet_search.invoke({"query": "What is LangChain?", "max_results": 3})
result

{'query': 'What is LangChain?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://aws.amazon.com/what-is/langchain/',
   'title': 'What is LangChain? - AWS',
   'content': 'LangChain is an open source framework for building applications based on large language models (LLMs). For example, developers can use LangChain components to build new prompt chains or customize existing templates. LangChain streamlines intermediate steps to develop such data-responsive applications, making prompt engineering more efficient. LangChain provides AI developers with tools to connect language models with external data sources. With LangChain, developers can adapt a language model flexibly to specific business contexts by designating steps required to produce the desired outcome. Developers then use the chain building blocks or LangChain Expression Language (LCEL) to compose chains with simple programming commands. Developers can create a prompt template for chat

In [12]:
# Tool calling
from langchain_openai import ChatOpenAI
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

Create an agent

In [13]:
from langchain.agents import create_agent


# System prompt to steer the agent to be smart assistent
AGENT_PROMPT = """You are a smart assistant. Answer user questions on the top 3 websites using the available tools.

- You have access to an internet_search tool as your primary means of gathering information.
- Use fetch_url to bring text content from the url.
- The answer should be clear and concise.
IMPORTANT:
Always include the 3 source URLs you used in your final answer.

"""

agent = create_agent(
    model=model_nemotron3_nano,
    tools=[internet_search,fetch_url],
    system_prompt=AGENT_PROMPT
)

In [14]:
from langchain_core.messages import HumanMessage

result = agent.invoke({
    "messages": [
        HumanMessage("What is the history of kingdom of Saudi Arabia ?")
    ]
})

In [15]:
# Print the agent's response
print(result["messages"][-1].content)

**Brief History of the Kingdom of Saudi Arabia**

| Period | Key Events |
|--------|------------|
| **Pre‑Islamic & Early Islamic Arabia** | The Arabian Peninsula was home to various nomadic tribes and the pre‑Islamic cultures of the Hejaz. In 610 CE the Prophet Muhammad received his first revelations in Mecca, leading to the spread of Islam across the region. |
| **First Saudi State (1744‑1818)** | The religious reformer **Muhammad ibn Saud** allied with **ʿUthmān ibn ʿAbdullāh** (the founder of the Wahhabi movement). Their partnership created the first Saudi‑Wahhabi state, which expanded across central Arabia but was eventually defeated by Ottoman forces in 1818. |
| **Second Saudi State (1824‑1891)** | **Imam Turki bin Abdullah** re‑established Saudi rule from Riyadh in 1824. The state flourished until internal rivalries and an Ottoman‑Egyptian invasion led to its collapse in 1891. |
| **Founding of the Modern Kingdom (1932)** | **Ibn Saud (Abdulaziz ibn Abdul‑Rahman Al Saud)** reca